# Structured V_θ on Multi-Xi Fock-PARFLM v2.1 — TinyStories

## Goal

Evaluate **structured V_θ variants** (mixture-of-Gaussians, quadratic well,
hybrid) as drop-in replacements for the MLP V_θ in the **Multi-channel ξ
Fock-PARFLM v2.1** (`FockMultiXiPARFLM`), targeting the **~8.95 PPL** baseline
achieved by the MLP-based model on TinyStories at 16k steps.

## Experiment cells

| Cell | V_θ variant | K_xi | Key params | Why interesting |
|------|------------|------|------------|----------------|
| `A1` | SQ3 K=4 (GMM, τ=1) | 4 | Modest mixture | Lower-capacity structured baseline |
| `A2` | **SQ3 K=8** (GMM, τ=1) | 4 | SPLM winner config | Primary contender |
| `A3` | SQ3 K=16 (GMM, τ=1) | 4 | Push mixture count | Does higher K help? |
| `A4` | SQ4 hybrid quad+MLP (h=64, d=2) | 4 | Safety net | Best of both worlds |
| `A5` | SQ2 low-rank (r=8) | 4 | Off-diagonal correlations | Does correlation structure help? |
| `B1` | **MLP baseline** (v_hidden=1024) | 4 | Reference | Reproduce the ~8.95 PPL baseline |

**Primary comparison:** A2 (SQ3 K=8, K_xi=4 — SPLM winner config) vs B1 (MLP, K_xi=4).

## Architecture

All cells use `FockMultiXiPARFLM` with d=256, L=8, fixed_gamma=0.3,
logfreq mass, xi_alpha_inits=[0.25, 0.50, 0.75, 0.95].
PARFLM: structural_competitive routing, top_k=8, Gumbel gates.
Fock v2.1: 16 registers, stack discipline, reverse channel, per-register τ/keys.
Only V_θ is swapped — all other dynamics are identical.

## 0. Environment setup + cell selector

In [ ]:
CELL = 'A1'       # one of: A1..A5, B1
SEED = 0

REPO_URL        = 'https://github.com/dimitarpg13/semsimula-paper.git'
REPO_BRANCH     = 'main'
COLAB_REPO_PATH = '/content/semsimula-paper'
GDRIVE_OUT_REL  = 'semsimula_fock_multixi_structured_vtheta'

import os, sys, shutil, subprocess, json, time, math
from pathlib import Path

os.environ.setdefault('PYTORCH_ALLOC_CONF', 'expandable_segments:True')

IN_COLAB = 'google.colab' in sys.modules
print(f'IN_COLAB = {IN_COLAB}')


def _sh(cmd: str) -> None:
    print(f'$ {cmd}')
    r = subprocess.run(cmd, shell=True)
    if r.returncode != 0:
        raise RuntimeError(f'command failed (exit {r.returncode}): {cmd}')


if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    GDRIVE_OUT = Path('/content/drive/MyDrive') / GDRIVE_OUT_REL
    GDRIVE_OUT.mkdir(parents=True, exist_ok=True)
    print(f'GDrive output root = {GDRIVE_OUT}')

    REPO_ROOT = Path(COLAB_REPO_PATH)
    if not (REPO_ROOT / '.git').exists():
        if REPO_ROOT.exists():
            shutil.rmtree(REPO_ROOT)
        _sh(
            f'git clone --depth 1 --branch {REPO_BRANCH} '
            f'{REPO_URL} {REPO_ROOT}'
        )
    else:
        try:
            _sh(f'git -C {REPO_ROOT} fetch --depth 1 origin {REPO_BRANCH}')
            _sh(f'git -C {REPO_ROOT} reset --hard origin/{REPO_BRANCH}')
        except RuntimeError as e:
            print(f'WARNING: could not refresh repo ({e}); using existing checkout.')

    DATA_CACHE = GDRIVE_OUT / 'data'
    DATA_CACHE.mkdir(exist_ok=True)
    repo_data_dir = REPO_ROOT / 'notebooks' / 'conservative_arch' / 'data'
    if repo_data_dir.is_symlink():
        repo_data_dir.unlink()
    elif repo_data_dir.is_dir():
        shutil.rmtree(repo_data_dir)
    repo_data_dir.symlink_to(DATA_CACHE)
    print(f'data/ -> {DATA_CACHE}')

    _sh('pip install -q transformers huggingface_hub pyarrow')
else:
    REPO_ROOT = Path('.').resolve()
    while not (REPO_ROOT / '.git').exists() and REPO_ROOT != REPO_ROOT.parent:
        REPO_ROOT = REPO_ROOT.parent
    GDRIVE_OUT = REPO_ROOT / 'notebooks' / 'conservative_arch' / 'scaleup' / 'results' / 'fock_multixi_structured_vtheta'
    GDRIVE_OUT.mkdir(parents=True, exist_ok=True)

SARF_DIR = REPO_ROOT / 'notebooks' / 'conservative_arch'
for sub in ['', 'multixi', 'parf', 'sarf_mass_variant', 'energetic_minima', 'scaleup']:
    d = str(SARF_DIR / sub) if sub else str(SARF_DIR)
    if d not in sys.path:
        sys.path.insert(0, d)

RESULTS_ROOT = GDRIVE_OUT
RUN_DIR = RESULTS_ROOT / CELL / f'seed{SEED}'
RUN_DIR.mkdir(parents=True, exist_ok=True)
print(f'Run output dir = {RUN_DIR}')

## 1. GPU check

In [ ]:
import torch
import numpy as np

if torch.cuda.is_available():
    device = 'cuda'
    props = torch.cuda.get_device_properties(0)
    total_memory = props.total_memory / 1e9
    print(f'GPU: {props.name}  ({total_memory:.1f} GB)')
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
else:
    device = 'cpu'
    total_memory = 0
    print('WARNING: no GPU detected \u2014 training will be very slow')

print(f'device = {device}')

## 2. Experiment recipes

In [ ]:
RECIPES = {
    # \u2500\u2500\u2500 Structured V_theta: SQ3 mixture \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
    'A1': {
        'desc': 'SQ3 K_mix=4, tau=1.0, K_xi=4',
        'v_theta_kind': 'sq3', 'K_mix': 4, 'tau': 1.0,
        'rank': None,
        'hybrid_v_hidden': None, 'hybrid_v_depth': None,
        'xi_channels': 4,
        'lambda_v': 1e-2,
    },
    'A2': {
        'desc': 'SQ3 K_mix=8, tau=1.0, K_xi=4 (primary \u2014 SPLM winner config)',
        'v_theta_kind': 'sq3', 'K_mix': 8, 'tau': 1.0,
        'rank': None,
        'hybrid_v_hidden': None, 'hybrid_v_depth': None,
        'xi_channels': 4,
        'lambda_v': 1e-2,
    },
    'A3': {
        'desc': 'SQ3 K_mix=16, tau=1.0, K_xi=4 (push mixture count higher)',
        'v_theta_kind': 'sq3', 'K_mix': 16, 'tau': 1.0,
        'rank': None,
        'hybrid_v_hidden': None, 'hybrid_v_depth': None,
        'xi_channels': 4,
        'lambda_v': 1e-2,
    },
    # \u2500\u2500\u2500 Structured V_theta: other variants \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
    'A4': {
        'desc': 'SQ4 hybrid quad + MLP (h=64, depth=2), K_xi=4',
        'v_theta_kind': 'sq4', 'K_mix': 1, 'tau': 1.0,
        'rank': None,
        'hybrid_v_hidden': 64, 'hybrid_v_depth': 2,
        'xi_channels': 4,
        'lambda_v': 1e-2,
    },
    'A5': {
        'desc': 'SQ2 low-rank (rank=8), K_xi=4',
        'v_theta_kind': 'sq2', 'K_mix': 1, 'tau': 1.0,
        'rank': 8,
        'hybrid_v_hidden': None, 'hybrid_v_depth': None,
        'xi_channels': 4,
        'lambda_v': 1e-2,
    },
    # \u2500\u2500\u2500 MLP baseline \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
    'B1': {
        'desc': 'MLP baseline v_hidden=1024 (~8.95 PPL reference), K_xi=4',
        'v_theta_kind': 'mlp', 'K_mix': None, 'tau': None,
        'rank': None,
        'hybrid_v_hidden': None, 'hybrid_v_depth': None,
        'xi_channels': 4,
        'lambda_v': 0.0,
    },
}

if CELL not in RECIPES:
    raise ValueError(f'CELL must be one of {sorted(RECIPES)}; got {CELL!r}')

recipe = RECIPES[CELL]

# \u2500\u2500\u2500 Shared architecture \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
D              = 256
L              = 8
V_HIDDEN       = 1024
V_DEPTH        = 3
VOCAB_SIZE     = 50257
MAX_LEN        = 1024
DT             = 1.0
FIXED_GAMMA    = 0.30
XI_CHANNELS    = recipe['xi_channels']
XI_LEARNABLE   = True
LAMBDA_V       = recipe['lambda_v']

STEPS          = 16000
BATCH          = 16
GRAD_ACCUM     = 1
BLOCK          = 512
LR             = 5e-4
WD             = 0.01
WARMUP         = 400
GRAD_CLIP      = 1.0
EVAL_INTERVAL  = 400
EVAL_ITERS     = 40
LOG_INTERVAL   = 50

if XI_CHANNELS == 4:
    XI_ALPHA_INITS = [0.25, 0.50, 0.75, 0.95]
elif XI_CHANNELS == 8:
    XI_ALPHA_INITS = [0.05, 0.15, 0.30, 0.50, 0.70, 0.85, 0.93, 0.98]
else:
    raise ValueError(f'No default alpha inits for xi_channels={XI_CHANNELS}')

print(f'Cell {CELL}: {recipe["desc"]}')
print(f'  V_theta = {recipe["v_theta_kind"]}')
if recipe['K_mix'] is not None:
    print(f'  K_mix = {recipe["K_mix"]}, tau = {recipe["tau"]}')
if recipe['rank'] is not None:
    print(f'  rank = {recipe["rank"]}')
if recipe['hybrid_v_hidden'] is not None:
    print(f'  hybrid MLP: hidden={recipe["hybrid_v_hidden"]}, depth={recipe["hybrid_v_depth"]}')
print(f'  xi_channels = {XI_CHANNELS}')
print(f'  xi_alpha_inits = {XI_ALPHA_INITS}')
print(f'  lambda_V = {LAMBDA_V}')
print(f'  steps={STEPS}  batch={BATCH}  block={BLOCK}  d={D}  L={L}')

## 3. Progress review (no GPU needed)

Safe to run without GPU model loaded. Reads Drive files only.

In [ ]:
import json, math
from pathlib import Path

_run_dir = RUN_DIR
_log_path = _run_dir / 'training_log.jsonl'

print('\u2500' * 55)
print(f'PROGRESS REVIEW  \u2014  Cell {CELL}: {recipe["desc"]}')
print('\u2500' * 55)

_best_ckpt = _run_dir / f'ckpt_best.pt'
if _best_ckpt.exists():
    try:
        _bd = torch.load(_best_ckpt, map_location='cpu', weights_only=False)
        print(f'\nBest checkpoint: PPL {_bd.get("val_ppl", float("nan")):.2f}'
              f'  at step {_bd.get("step", 0):,}')
    except Exception:
        pass

_latest_ckpt = _run_dir / f'ckpt_latest.pt'
if _latest_ckpt.exists():
    try:
        _ld = torch.load(_latest_ckpt, map_location='cpu', weights_only=False)
        print(f'Latest checkpoint: step {_ld.get("step", 0):,}'
              f'  PPL {_ld.get("val_ppl", float("nan")):.2f}')
    except Exception:
        pass

eval_entries = []
if _log_path.exists():
    with open(_log_path) as f:
        for line in f:
            try:
                e = json.loads(line)
                if 'val_ppl' in e:
                    eval_entries.append((e['step'], e['val_ppl']))
            except Exception:
                pass
if eval_entries:
    print(f'\nVal PPL history ({len(eval_entries)} evals):')
    best_s, best_p = min(eval_entries, key=lambda x: x[1])
    last_s, last_p = eval_entries[-1]
    for step, ppl in eval_entries[-10:]:
        marker = ' \u2190 best' if (step, ppl) == (best_s, best_p) else ''
        print(f'  step {step:>6,}:  PPL {ppl:>8.2f}{marker}')
    if len(eval_entries) > 10:
        print(f'  ... ({len(eval_entries) - 10} earlier entries omitted)')
    print(f'\n  Best PPL : {best_p:.2f}  at step {best_s:,}')
    print(f'  Latest   : {last_p:.2f}  at step {last_s:,}')
    print(f'  Progress : {last_s:,} / {STEPS:,} ({100*last_s/STEPS:.1f}%)')

    bars = ' \u2581\u2582\u2583\u2584\u2585\u2586\u2587\u2588'
    ppls = [p for _, p in eval_entries[-60:]]
    lo, hi = min(ppls), max(ppls)
    rng = hi - lo if hi > lo else 1.0
    spark = ''.join(bars[min(8, int(8 * (p - lo) / rng))] for p in ppls)
    print(f'  Sparkline: {spark}  ({lo:.1f} \u2500 {hi:.1f})')
else:
    print('\n  No training data yet.')

print('\n' + '\u2500' * 55)

## 4. Load TinyStories

In [ ]:
from data_module import load_tiny_stories, get_batch

train_ids, val_ids = load_tiny_stories(max_train_tokens=5_000_000)
print(f'train: {len(train_ids):,} tokens   val: {len(val_ids):,} tokens')

rng = np.random.default_rng(SEED)

## 5. Build model + V_θ swap

The Fock-PARFLM v2.1 V_θ takes `(xis, h)` where `xis` is `(B, T, K, d)`.
The structured V_θ classes take `(xi, h)` where `xi` is `(..., d)`.
We use `StructuredVThetaMultiXiAdapter` to bridge the interface:
it flattens the K xi-channels into a single `(K+1)*d` vector and passes
it as the `xi` input to the structured V_θ, which is widened accordingly.

In [ ]:
from model_fock_parf_multixi import FockMultiXiPARFLM, FockMultiXiPARFConfig
import model_fock_parf_v2
import model_parf_multixi
import model_parf
import model_parf_sparse
from model_structured_vtheta import (
    MixtureQuadraticVTheta, QuadraticWellVTheta,
    LowRankQuadraticVTheta, HybridQuadraticVTheta,
    validate_analytical_grad, StructuredVThetaBase,
)
import torch.nn.functional as F_torch


class StructuredVThetaMultiXiAdapter(torch.nn.Module):
    """Adapts a structured V_theta (single-xi interface) to the Multi-Xi
    V_theta interface: (xis: (B,T,K,d), h: (B,T,d)) -> V: (B,T,1).

    The K xi-channels are flattened and concatenated with h, then split
    so the structured V_theta sees xi_flat as its 'xi' input.
    This means the structured V_theta operates on dim = (K+1)*d total
    input, where the 'xi' portion is K*d and the 'h' portion is d.
    """

    def __init__(self, inner: StructuredVThetaBase, K: int, d: int):
        super().__init__()
        self.inner = inner
        self.K = K
        self.d = d

    def forward(self, xis: torch.Tensor, h: torch.Tensor) -> torch.Tensor:
        B, T, K, d = xis.shape
        xi_flat = xis.reshape(B, T, K * d)
        return self.inner(xi_flat, h)

    def analytical_grad(self, xis: torch.Tensor, h: torch.Tensor) -> torch.Tensor:
        B, T, K, d = xis.shape
        xi_flat = xis.reshape(B, T, K * d)
        return self.inner.analytical_grad(xi_flat, h)

    def attractor_centres(self, xis: torch.Tensor) -> torch.Tensor:
        B, T, K, d = xis.shape
        xi_flat = xis.reshape(B, T, K * d)
        return self.inner.attractor_centres(xi_flat)


# \u2500\u2500 Logfreq surprisal \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
LOGFREQ_PATH = SARF_DIR / 'scaleup' / 'results' / 'logfreq_surprisal_tinystories.npy'
DRIVE_LOGFREQ = RESULTS_ROOT / 'logfreq_surprisal_tinystories.npy'

if LOGFREQ_PATH.exists():
    LOGFREQ_FILE = LOGFREQ_PATH
    print(f'Using bundled logfreq: {LOGFREQ_FILE}')
elif DRIVE_LOGFREQ.exists():
    LOGFREQ_FILE = DRIVE_LOGFREQ
    print(f'Using Drive-cached logfreq: {LOGFREQ_FILE}')
else:
    counts = np.bincount(train_ids.astype(np.int64), minlength=VOCAB_SIZE).astype(np.float64)
    p = (counts + 1.0) / (counts.sum() + VOCAB_SIZE)
    surprisal = (-np.log(p)).astype(np.float32)
    LOGFREQ_FILE = DRIVE_LOGFREQ
    LOGFREQ_FILE.parent.mkdir(parents=True, exist_ok=True)
    np.save(LOGFREQ_FILE, surprisal)
    print(f'Built logfreq from train_ids; saved to {LOGFREQ_FILE}')

# \u2500\u2500 Build model \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
torch.manual_seed(SEED)

cfg = FockMultiXiPARFConfig(
    vocab_size=VOCAB_SIZE, d=D, max_len=MAX_LEN,
    L=L, v_hidden=V_HIDDEN, v_depth=V_DEPTH, dt=DT,
    mass_mode='logfreq',
    logfreq_path=str(LOGFREQ_FILE),
    logfreq_init_alpha=0.1,
    init_gamma=1.0,
    fixed_gamma=FIXED_GAMMA,
    causal_force=True,
    ln_after_step=True,
    xi_channels=XI_CHANNELS,
    xi_alpha_inits=XI_ALPHA_INITS,
    xi_learnable=XI_LEARNABLE,
    xi_alpha_init_mode='explicit',
    # PARFLM-specific
    v_phi_kind='structural_competitive',
    v_phi_phi_hidden=128,
    v_phi_theta_hidden=128,
    top_k=8,
    score_head_hidden=32,
    gumbel_tau_init=1.0,
    gumbel_tau_min=0.3,
    gumbel_noise=True,
    use_gathered_v_phi=True,
    use_layer_checkpoint=True,
    ln_before_distance=True,
    per_layer_v_phi_scale=True,
    # Fock v2.1-specific
    fock_version='v2',
    n_registers=16,
    register_salience_decay=0.5,
    register_salience_threshold=0.005,
    creation_gate_hidden=64,
    stack_discipline=True,
    d_k=64,
    tau_create_init=8.0,
    reverse_channel=True,
    per_register_tau=True,
    per_register_keys=True,
    ortho_register_init=True,
    prefix_causal_registers=True,  # causal leak fix — set False only to reproduce leaky baselines
)
model = FockMultiXiPARFLM(cfg).to(device)

n_total_before = sum(p.numel() for p in model.parameters())
n_v_theta_before = sum(p.numel() for p in model.V_theta.parameters())
print(f'Before swap: total={n_total_before:,}  V_theta={n_v_theta_before:,}')

# \u2500\u2500 V_theta swap \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
vkind = recipe['v_theta_kind']
xi_d = XI_CHANNELS * D

if vkind == 'sq3':
    inner = MixtureQuadraticVTheta.__new__(MixtureQuadraticVTheta)
    torch.nn.Module.__init__(inner)
    inner.d = D
    inner.K = recipe['K_mix']
    inner.tau = recipe['tau']
    inner.mu_proj = torch.nn.Linear(xi_d, recipe['K_mix'] * D)
    inner.a_proj = torch.nn.Linear(xi_d, recipe['K_mix'] * D)
    inner.pi_proj = torch.nn.Linear(xi_d, recipe['K_mix'])
    inner.b_proj = torch.nn.Linear(xi_d, 1)
    inner._init_weights(0.0)
    adapter = StructuredVThetaMultiXiAdapter(inner, K=XI_CHANNELS, d=D).to(device)
    model.V_theta = adapter
    print(f'[{CELL}] V_theta -> SQ3 Mixture(K_mix={recipe["K_mix"]}, tau={recipe["tau"]}) via MultiXi adapter')

elif vkind == 'sq1':
    inner = QuadraticWellVTheta.__new__(QuadraticWellVTheta)
    torch.nn.Module.__init__(inner)
    inner.d = D
    inner.mu_proj = torch.nn.Linear(xi_d, D)
    inner.a_proj = torch.nn.Linear(xi_d, D)
    inner.b_proj = torch.nn.Linear(xi_d, 1)
    inner._init_weights(0.0)
    adapter = StructuredVThetaMultiXiAdapter(inner, K=XI_CHANNELS, d=D).to(device)
    model.V_theta = adapter
    print(f'[{CELL}] V_theta -> SQ1 QuadraticWell via MultiXi adapter')

elif vkind == 'sq2':
    inner = LowRankQuadraticVTheta.__new__(LowRankQuadraticVTheta)
    torch.nn.Module.__init__(inner)
    inner.d = D
    inner.rank = recipe['rank']
    inner.mu_proj = torch.nn.Linear(xi_d, D)
    inner.lam_proj = torch.nn.Linear(xi_d, D)
    inner.U_proj = torch.nn.Linear(xi_d, D * recipe['rank'])
    inner.b_proj = torch.nn.Linear(xi_d, 1)
    inner._init_weights(0.0)
    adapter = StructuredVThetaMultiXiAdapter(inner, K=XI_CHANNELS, d=D).to(device)
    model.V_theta = adapter
    print(f'[{CELL}] V_theta -> SQ2 LowRank(rank={recipe["rank"]}) via MultiXi adapter')

elif vkind == 'sq4':
    inner = HybridQuadraticVTheta.__new__(HybridQuadraticVTheta)
    torch.nn.Module.__init__(inner)
    inner.d = D
    inner.quad = QuadraticWellVTheta.__new__(QuadraticWellVTheta)
    torch.nn.Module.__init__(inner.quad)
    inner.quad.d = D
    inner.quad.mu_proj = torch.nn.Linear(xi_d, D)
    inner.quad.a_proj = torch.nn.Linear(xi_d, D)
    inner.quad.b_proj = torch.nn.Linear(xi_d, 1)
    inner.quad._init_weights(0.0)
    h_h, h_d = recipe['hybrid_v_hidden'], recipe['hybrid_v_depth']
    layers = [torch.nn.Linear(xi_d + D, h_h), torch.nn.GELU()]
    for _ in range(h_d - 1):
        layers += [torch.nn.Linear(h_h, h_h), torch.nn.GELU()]
    layers += [torch.nn.Linear(h_h, 1)]
    inner.mlp = torch.nn.Sequential(*layers)
    for m in inner.mlp.modules():
        if isinstance(m, torch.nn.Linear):
            torch.nn.init.normal_(m.weight, std=0.02)
            if m.bias is not None:
                torch.nn.init.zeros_(m.bias)
    inner.alpha = torch.nn.Parameter(torch.tensor(0.1))
    adapter = StructuredVThetaMultiXiAdapter(inner, K=XI_CHANNELS, d=D).to(device)
    model.V_theta = adapter
    print(f'[{CELL}] V_theta -> SQ4 Hybrid(h={h_h}, depth={h_d}) via MultiXi adapter')

else:
    print(f'[{CELL}] Keeping MLP V_theta (v_hidden={V_HIDDEN}, v_depth={V_DEPTH})')

# \u2500\u2500 Report \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
n_total = sum(p.numel() for p in model.parameters())
n_v_theta = sum(p.numel() for p in model.V_theta.parameters())
print(f'\nAfter swap:')
print(f'  total params   = {n_total:,}')
print(f'  V_theta params = {n_v_theta:,}  ({n_v_theta/n_total*100:.1f}%)')
print(f'  V_theta reduction vs MLP: {n_v_theta_before:,} -> {n_v_theta:,} '
      f'({n_v_theta_before/max(n_v_theta,1):.0f}x)')
print(f'  xi_alpha init: {model.xi_alpha_values()}')

IS_STRUCTURED = vkind != 'mlp'
IS_MIXTURE = vkind == 'sq3'

## 6. Training loop

In [ ]:
def lr_at(step):
    if step < WARMUP:
        return LR * (step + 1) / WARMUP
    progress = (step - WARMUP) / max(STEPS - WARMUP, 1)
    return LR * 0.5 * (1.0 + math.cos(math.pi * min(progress, 1.0)))


def forward_with_vreg(model, x, targets, lambda_v):
    h0 = model._embed(x)
    h_L, _ = model._stack_forward(h0, x, return_trajectory=False)
    logits = h_L @ model.E.weight.T
    loss_ntp = F_torch.cross_entropy(
        logits.reshape(-1, cfg.vocab_size),
        targets.reshape(-1),
    )

    v_reg_value = torch.tensor(0.0, device=x.device)
    if lambda_v > 0 and IS_STRUCTURED:
        xis = model.xi_module(h_L.detach())
        V_vals = model.V_theta(xis, h_L)
        v_reg_value = (V_vals ** 2).mean()
        loss = loss_ntp + lambda_v * v_reg_value
    else:
        loss = loss_ntp

    return logits, loss, loss_ntp, v_reg_value


@torch.no_grad()
def evaluate_model():
    model.eval()
    losses = []
    for _ in range(EVAL_ITERS):
        xb, yb = get_batch(val_ids, BATCH, BLOCK, rng)
        x = torch.from_numpy(xb).to(device)
        y = torch.from_numpy(yb).to(device)
        with torch.enable_grad():
            _, loss = model(x, y)
        losses.append(loss.item())
    model.train()
    return float(np.mean(losses))


resume_step = 0
_latest_ckpt_path = RUN_DIR / 'ckpt_latest.pt'
_best_ckpt_path = RUN_DIR / 'ckpt_best.pt'

opt = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=LR, betas=(0.9, 0.95), weight_decay=WD,
)

if _latest_ckpt_path.exists():
    ckpt = torch.load(_latest_ckpt_path, map_location=device, weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'])
    if 'optimizer_state_dict' in ckpt:
        opt.load_state_dict(ckpt['optimizer_state_dict'])
    resume_step = ckpt.get('step', 0)
    print(f'Resumed from step {resume_step:,}  (PPL {ckpt.get("val_ppl", "?")})  [{_latest_ckpt_path.name}]')

best_val_ppl = float('inf')
if _best_ckpt_path.exists():
    try:
        _bd = torch.load(_best_ckpt_path, map_location='cpu', weights_only=False)
        best_val_ppl = _bd.get('val_ppl', float('inf'))
        print(f'Restored best PPL from previous session: {best_val_ppl:.2f}')
        del _bd
    except Exception as e:
        print(f'[warn] Could not read best checkpoint: {e}')

log_path = RUN_DIR / 'training_log.jsonl'
log_f = log_path.open('a')

# ── Causal leak monitoring knobs ──────────────────────────────────────
CAUSAL_PROBE_INTERVAL = 4000   # architectural probe every N steps (0=off)
SPIKE_THRESHOLD       = 500.0  # pre-clip grad norm that counts as a spike (high for TinyStories)
SPIKE_COOLDOWN        = 20     # min steps between logged spikes
_last_spike_step      = -10**9

def _log_write(record):
    log_f.write(json.dumps(record) + '\n')
    log_f.flush()

def run_causal_probe(step_num):
    """Lightweight architectural causal probe on CPU in float64."""
    from model_fock_parf_multixi import FockMultiXiPARFConfig as _ProbeCfg
    from model_fock_parf_multixi import FockMultiXiPARFLM as _ProbeModel
    _probe_cfg = _ProbeCfg(
        vocab_size=256, d=32, max_len=64, L=2,
        v_hidden=64, v_depth=1, dt=0.1,
        mass_mode='uniform', causal_force=True,
        ln_after_step=True,
        xi_channels=cfg.xi_channels,
        xi_alpha_inits=[0.0] * cfg.xi_channels,
        xi_learnable=False, xi_alpha_init_mode='explicit',
        fock_version='v2', n_registers=4,
        register_salience_decay=0.5,
        register_salience_threshold=0.005,
        creation_gate_hidden=16, stack_discipline=True,
        d_k=16, tau_create_init=8.0,
        reverse_channel=True,
        prefix_causal_registers=getattr(cfg, 'prefix_causal_registers', True),
        per_register_tau=True, per_register_keys=True,
        ortho_register_init=True,
    )
    _m = _ProbeModel(_probe_cfg).double().cpu()
    with torch.no_grad():
        for n, p in _m.named_parameters():
            if 'reverse_channel_scale' in n:
                p.fill_(5.0)
    _ids = torch.randint(0, 256, (1, 32))
    results = {}
    for mode_name, use_train in [('eval', False), ('train', True)]:
        if use_train:
            _m.train()
        else:
            _m.eval()
        with torch.no_grad():
            logits_clean, _ = _m(_ids)
        _ids_pert = _ids.clone()
        _ids_pert[0, 20:] = torch.randint(0, 256, (12,))
        with torch.no_grad():
            logits_pert, _ = _m(_ids_pert)
        delta = (logits_pert[0, :20] - logits_clean[0, :20]).abs().max().item()
        results[mode_name] = delta
    passed = results['eval'] == 0.0 and results['train'] == 0.0
    status = 'PASS' if passed else 'FAIL'
    print(f'[causal-probe] step {step_num}: {status}  '
          f'eval_delta={results["eval"]:.2e}  train_delta={results["train"]:.2e}')
    _log_write({
        'step': step_num, 'event': 'causal_probe',
        'eval_max_delta': results['eval'],
        'train_max_delta': results['train'],
        'passed': passed,
    })
    del _m
    return passed

model.train()
t0 = time.time()
t_session = time.time()

for step in range(resume_step, STEPS):
    for g in opt.param_groups:
        g['lr'] = lr_at(step)

    opt.zero_grad(set_to_none=True)
    step_loss_ntp = 0.0
    step_v_reg = 0.0
    step_loss_total = 0.0

    for _acc in range(GRAD_ACCUM):
        xb, yb = get_batch(train_ids, BATCH, BLOCK, rng)
        x = torch.from_numpy(xb).to(device)
        y = torch.from_numpy(yb).to(device)
        _, loss, loss_ntp, v_reg = forward_with_vreg(model, x, y, LAMBDA_V)
        (loss / GRAD_ACCUM).backward()
        step_loss_ntp   += loss_ntp.item() / GRAD_ACCUM
        step_v_reg      += v_reg.item()    / GRAD_ACCUM
        step_loss_total += loss.item()     / GRAD_ACCUM

    grad_norm = torch.nn.utils.clip_grad_norm_(
        [p for p in model.parameters() if p.requires_grad], GRAD_CLIP,
    ).item()
    opt.step()

    # ── Spike detection + JSONL logging ──
    if (grad_norm > SPIKE_THRESHOLD
            and (step - _last_spike_step) >= SPIKE_COOLDOWN):
        _last_spike_step = step
        print(f'\n[spike] step {step+1}: pre-clip grad={grad_norm:.1f}  '
              f'ntp={step_loss_ntp:.4f}  v_reg={step_v_reg:.4f}')
        _log_write({
            'step': step + 1, 'event': 'grad_spike',
            'pre_clip_grad_norm': round(grad_norm, 2),
            'ntp': round(step_loss_ntp, 4),
            'v_reg': round(step_v_reg, 4),
        })

    # ── Periodic causal probe ──
    if (CAUSAL_PROBE_INTERVAL > 0
            and (step + 1) % CAUSAL_PROBE_INTERVAL == 0):
        run_causal_probe(step + 1)

    if (step + 1) % LOG_INTERVAL == 0 or step == 0:
        elapsed = time.time() - t_session
        steps_done = step + 1 - resume_step
        sec_per_step = elapsed / max(steps_done, 1)
        remaining = (STEPS - step - 1) * sec_per_step
        alphas_str = ','.join(f'{a:.3f}' for a in model.xi_alpha_values())
        print(f'[{CELL}] step {step+1:>5}/{STEPS}  '
              f'ntp={step_loss_ntp:.4f}  v_reg={step_v_reg:.4f}  '
              f'lr={lr_at(step):.2e}  grad={grad_norm:.3f}  '
              f'gamma={model.gamma.item():.3f}  '
              f'alpha=[{alphas_str}]  '
              f'{elapsed:.0f}s (~{remaining/60:.1f}m remaining)')
        log_entry = {
            'step': step + 1, 'train_loss': step_loss_ntp,
            'v_reg': step_v_reg, 'total_loss': step_loss_total,
            'lr': lr_at(step), 'grad_norm': grad_norm,
            'gamma': model.gamma.item(),
            'xi_alphas': model.xi_alpha_values(),
        }
        log_f.write(json.dumps(log_entry) + '\n')
        log_f.flush()

    if (step + 1) % EVAL_INTERVAL == 0 or (step + 1) == STEPS:
        val_loss = evaluate_model()
        val_ppl = math.exp(val_loss)
        is_best = val_ppl < best_val_ppl
        if is_best:
            best_val_ppl = val_ppl
        best_marker = '  *** NEW BEST ***' if is_best else ''
        print(f'  >> val_loss={val_loss:.4f}  val_ppl={val_ppl:.2f}  '
              f'best={best_val_ppl:.2f}{best_marker}')

        eval_entry = {
            'step': step + 1, 'val_loss': val_loss,
            'val_ppl': val_ppl, 'best_ppl': best_val_ppl,
        }
        log_f.write(json.dumps(eval_entry) + '\n')
        log_f.flush()

        if is_best:
            _best_state = {
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': opt.state_dict(),
                'step': step + 1, 'val_loss': val_loss,
                'val_ppl': val_ppl, 'gamma': model.gamma.item(),
                'xi_alphas': model.xi_alpha_values(),
                'cell': CELL, 'recipe': recipe, 'seed': SEED,
            }
            torch.save(_best_state, _best_ckpt_path)

    if (step + 1) % 4000 == 0 or (step + 1) == STEPS:
        _ckpt = {
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': opt.state_dict(),
            'step': step + 1, 'val_ppl': val_ppl if (step+1) % EVAL_INTERVAL == 0 else best_val_ppl,
            'best_val_ppl': best_val_ppl,
            'gamma': model.gamma.item(),
            'xi_alphas': model.xi_alpha_values(),
            'cell': CELL, 'recipe': recipe, 'seed': SEED,
        }
        torch.save(_ckpt, _latest_ckpt_path)
        print(f'  [ckpt] saved {_latest_ckpt_path.name} at step {step+1}')

log_f.close()
print(f'\n[{CELL}] Training done.  total wall = {time.time()-t_session:.0f}s  '
      f'final_ppl = {val_ppl:.2f}  best_ppl = {best_val_ppl:.2f}')

## 7. Training curve

In [ ]:
import matplotlib.pyplot as plt

eval_entries = []
if log_path.exists():
    with open(log_path) as f:
        for line in f:
            try:
                e = json.loads(line)
                if 'val_ppl' in e:
                    eval_entries.append(e)
            except Exception:
                pass

if eval_entries:
    steps_arr = [e['step'] for e in eval_entries]
    ppls = [e['val_ppl'] for e in eval_entries]

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(steps_arr, ppls, 'o-', label=f'{CELL}: {recipe["desc"]}', linewidth=1.5)
    ax.axhline(y=8.95, color='red', linestyle='--', alpha=0.7,
               label='Fock-PARFLM v2.1 MLP baseline (8.95 PPL)')
    ax.set_xlabel('Step')
    ax.set_ylabel('Val PPL')
    ax.set_title(f'Structured V_\u03b8 on Multi-Xi Fock-PARFLM v2.1 \u2014 {CELL}')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    fig.savefig(RUN_DIR / f'training_curve_{CELL}.png', dpi=150)
    plt.show()
    print(f'Saved: {RUN_DIR / f"training_curve_{CELL}.png"}')
else:
    print('No eval data to plot.')

## 8. V_\u03b8 landscape diagnostics

In [ ]:
v_samples = []
model.eval()
for _ in range(10):
    xb, _ = get_batch(val_ids, BATCH, BLOCK, rng)
    x = torch.from_numpy(xb).to(device)
    with torch.enable_grad():
        h0 = model._embed(x)
        h_L, _ = model._stack_forward(h0, x, return_trajectory=False)
        xis = model.xi_module(h_L.detach())
        V_vals = model.V_theta(xis, h_L)
        v_samples.append(V_vals.detach().cpu().numpy().ravel())

v_all = np.concatenate(v_samples)
ls = {
    'mean': float(v_all.mean()),
    'std': float(v_all.std()),
    'min': float(v_all.min()),
    'max': float(v_all.max()),
    'range': float(v_all.max() - v_all.min()),
}
print(f'V_theta landscape stats:')
for k, v in ls.items():
    print(f'  {k:6s}: {v:.4f}')

ls_path = RUN_DIR / f'landscape_stats_{CELL}.json'
with open(ls_path, 'w') as f:
    json.dump(ls, f, indent=2)

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(v_all, bins=80, edgecolor='none', alpha=0.8)
ax.axvline(ls['mean'], color='red', linestyle='--', alpha=0.6, label=f'mean={ls["mean"]:.2f}')
ax.set_xlabel('V_\u03b8(\u03be, h)')
ax.set_ylabel('count')
ax.set_title(f'{CELL} V_\u03b8 distribution ({recipe["desc"]})')
ax.legend()
plt.tight_layout()
fig.savefig(RUN_DIR / f'v_theta_hist_{CELL}.png', dpi=150)
plt.show()
model.train()

## 9. Attractor analysis (SQ3 only)

For mixture variants, the K attractor centres mu_k(xi) are read directly
from the model parameters — no gradient descent needed.

In [ ]:
if IS_MIXTURE:
    print(f'=== Analytical attractor centres (K_mix={recipe["K_mix"]}) ===')
    model.eval()
    try:
        from transformers import GPT2Tokenizer
        tok = GPT2Tokenizer.from_pretrained('gpt2')
    except Exception as e:
        print(f'Tokenizer unavailable ({e}); skipping centroid decoding')
        tok = None

    PROMPTS = {
        'narrative': 'Once upon a time in a small village there lived',
        'dialogue': 'The little girl said to her mother',
        'description': 'The garden was full of beautiful flowers and',
        'action': 'The boy ran as fast as he could toward',
        'emotion': 'She felt very happy because her friend gave her',
    }

    attractor_data = {}
    for pname, text in PROMPTS.items():
        if tok is None:
            continue
        ids = tok.encode(text)
        x = torch.tensor([ids], device=device)
        with torch.no_grad():
            h0 = model._embed(x)
            h_L, _ = model._stack_forward(h0, x, return_trajectory=False)
            xis = model.xi_module(h_L.detach())
            centres = model.V_theta.attractor_centres(xis)

        centres_np = centres[0].detach().cpu().numpy()  # (T, K_mix, d)
        last_centres = centres_np[-1]  # (K_mix, d)

        decoded = []
        for k in range(recipe['K_mix']):
            c = torch.tensor(last_centres[k], device=device, dtype=torch.float32)
            logits = c @ model.E.weight.T
            probs = torch.softmax(logits, dim=-1)
            top5_vals, top5_ids = probs.topk(5)
            decoded.append([
                (tok.decode([tid.item()]), top5_vals[j].item())
                for j, tid in enumerate(top5_ids)
            ])

        attractor_data[pname] = {
            'K': recipe['K_mix'], 'decoded': decoded, 'method': 'analytical',
        }

        print(f'\nPrompt: "{pname}"')
        for k, top_tokens in enumerate(decoded):
            top3 = [f'"{ t }"({p:.1e})' for t, p in top_tokens[:3]]
            print(f'  Basin {k+1}: {", ".join(top3)}')

    if attractor_data:
        with open(RUN_DIR / f'attractors_{CELL}.json', 'w') as f:
            json.dump(attractor_data, f, indent=2)
        print(f'\nSaved attractor data to {RUN_DIR / f"attractors_{CELL}.json"}')
    model.train()
else:
    print('Attractor analysis only available for SQ3 mixture variants.')

## 10. Cross-cell comparison dashboard

Run after completing multiple cells to see the full comparison.

In [ ]:
ALL_CELLS = sorted(RECIPES.keys())

dashboard = {}
for cell_name in ALL_CELLS:
    cell_dir = RESULTS_ROOT / cell_name / f'seed{SEED}'
    if not cell_dir.exists():
        dashboard[cell_name] = None
        continue
    log_file = cell_dir / 'training_log.jsonl'
    if not log_file.exists():
        dashboard[cell_name] = None
        continue
    evals = []
    with open(log_file) as f:
        for line in f:
            try:
                e = json.loads(line)
                if 'val_ppl' in e:
                    evals.append(e)
            except Exception:
                pass
    if not evals:
        dashboard[cell_name] = None
        continue
    best_ppl = min(e['val_ppl'] for e in evals)
    final_ppl = evals[-1]['val_ppl']
    last_step = evals[-1]['step']

    ls_files = sorted(cell_dir.glob(f'landscape_stats_*.json'))
    ls = json.loads(ls_files[-1].read_text()) if ls_files else None

    dashboard[cell_name] = {
        'desc': RECIPES[cell_name]['desc'],
        'best_ppl': best_ppl, 'final_ppl': final_ppl,
        'last_step': last_step, 'landscape': ls,
    }

print(f'{"Cell":<6} {"Description":<52} {"best PPL":>10} {"final PPL":>10} {"V range":>8}')
print('\u2500' * 92)
print(f'{"":6} {"MLP baseline (8.95 PPL, 16k steps)":<52} {"8.95":>10} {"\u2014":>10} {"\u2014":>8}')
print('\u2500' * 92)

for cell_name in ALL_CELLS:
    r = dashboard[cell_name]
    desc = RECIPES[cell_name]['desc'][:50]
    if r is None:
        print(f'{cell_name:<6} {desc:<52} {"\u2014":>10} {"\u2014":>10} {"\u2014":>8}    (not run)')
    else:
        v_range = f"{r['landscape']['range']:.1f}" if r['landscape'] else '\u2014'
        print(f'{cell_name:<6} {desc:<52} {r["best_ppl"]:>10.2f} {r["final_ppl"]:>10.2f} {v_range:>8}')

## 11. Save summary

Save a human-readable summary of this cell's results.

In [ ]:
summary_path = RUN_DIR / f'summary_{CELL}.md'
with open(summary_path, 'w') as f:
    f.write(f'# {CELL}: {recipe["desc"]}\n\n')
    f.write(f'| Setting | Value |\n')
    f.write(f'|---------|-------|\n')
    f.write(f'| V_theta kind | {recipe["v_theta_kind"]} |\n')
    if recipe['K_mix'] is not None:
        f.write(f'| K_mix | {recipe["K_mix"]} |\n')
        f.write(f'| tau | {recipe["tau"]} |\n')
    f.write(f'| xi_channels | {XI_CHANNELS} |\n')
    f.write(f'| lambda_V | {LAMBDA_V} |\n')
    f.write(f'| d | {D} |\n')
    f.write(f'| L | {L} |\n')
    f.write(f'| steps | {STEPS} |\n')
    f.write(f'| best PPL | {best_val_ppl:.2f} |\n')
    f.write(f'| V_theta params | {n_v_theta:,} |\n')
    f.write(f'| total params | {n_total:,} |\n')
    f.write(f'| xi_alpha_init | {XI_ALPHA_INITS} |\n')
    f.write(f'\n## Reference\n\n')
    f.write(f'Fock-PARFLM v2.1 MLP baseline (K_xi=4, 16k steps): **8.95 PPL**\n')

print(f'Summary saved to {summary_path}')
print(f'\nFinal result: {CELL} = {best_val_ppl:.2f} PPL  (Fock-PARFLM v2.1 MLP baseline = 8.95)')